# **AI TECH INSTITUTE** · *Intermediate AI & Data Science*
### Week 8 - Template Notebook: Machine Learning Model Lifecycle
**Instructor:** Amir Charkhi | **Goal:** Full ML Pipeline: From Data to Deployment


### Learning Objectives
- Understand the complete ML workflow from data loading to model evaluation
- Learn proper data splitting to avoid data leakage
- Compare linear and tree-based models
- Master cross-validation and hyperparameter tuning
- Apply best practices for model evaluation

---

## 1. Import Libraries

**What you need to do:**  
Import all necessary libraries for data manipulation, visualization, and machine learning.

**Required imports:**
- NumPy and Pandas for data handling
- Matplotlib and Seaborn for visualization
- Scikit-learn for dataset, preprocessing, models, and evaluation

**💡 Hint:** Import `train_test_split`, `LinearRegression`, `DecisionTreeRegressor`, `cross_val_score`, `GridSearchCV`, and regression metrics.

In [ ]:
# Import all necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
print("Libraries imported successfully!")


---
## 2. Load the Dataset

**What you need to do:**  
Load the California Housing dataset using sklearn's built-in dataset.

**Theory:**  
The California Housing dataset contains information from the 1990 census with features like median income, house age, and location. The target variable is the median house value.

**💡 Hint:** Use `fetch_california_housing()` and convert to a pandas DataFrame. Set `as_frame=True` for easy handling.

In [ ]:
# Load the California Housing dataset
housing = fetch_california_housing(as_frame=True)
X = housing.data
y = housing.target

print(f"Dataset loaded: {len(X)} samples, {X.shape[1]} features")
print(f"Target variable: {housing.target_names}")


---
## 3. Initial Data Inspection

**What you need to do:**  
Perform a quick inspection of the dataset before any splitting.

**Tasks:**
- Display the first few rows
- Check dataset shape
- Display feature names and target variable
- Check for missing values

**💡 Hint:** Use `.head()`, `.shape`, `.info()`, and `.isnull().sum()` methods.

In [ ]:
# Inspect the dataset structure
print("First 5 rows:")
print(X.head())

print(f"\nDataset shape: {X.shape}")
print(f"\nFeature names: {list(X.columns)}")

print("\nMissing values:")
print(X.isnull().sum())

print("\nTarget variable statistics:")
print(f"Min: {y.min():.2f}, Max: {y.max():.2f}, Mean: {y.mean():.2f}")


---
## 4. Train-Validation-Test Split

**⚠️ CRITICAL: Split BEFORE detailed EDA to prevent data leakage!**

**What you need to do:**  
Split the data into three sets:
- **Training set (60%)**: For model training
- **Validation set (20%)**: For model selection and hyperparameter tuning
- **Test set (20%)**: For final, unbiased evaluation (DO NOT TOUCH until the very end!)

**Theory:**  
The test set represents unseen data in production. It must remain completely isolated from all training decisions to give an honest estimate of model performance.

**💡 Hint:** Use `train_test_split()` twice. First split into train+val (80%) and test (20%), then split train+val into train (75% of 80% = 60% total) and validation (25% of 80% = 20% total). Set `random_state=42` for reproducibility.

In [ ]:
# Split data into train, validation, and test sets
# First split: 80% train+val, 20% test
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Second split: 75% of temp = 60% train, 25% of temp = 20% val
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42)

print(f"Training set: {len(X_train)} samples ({len(X_train)/len(X)*100:.0f}%)")
print(f"Validation set: {len(X_val)} samples ({len(X_val)/len(X)*100:.0f}%)")
print(f"Test set: {len(X_test)} samples ({len(X_test)/len(X)*100:.0f}%)")
print("\nTest set is locked until final evaluation!")


---
## 5. Exploratory Data Analysis (EDA)

**⚠️ IMPORTANT: Perform EDA ONLY on the training set to avoid data leakage!**

**What you need to do:**  
Analyze the training data to understand patterns, distributions, and relationships.

**Tasks:**
1. Display summary statistics for all features
2. Visualize target variable distribution (histogram)
3. Create a correlation heatmap
4. Identify the top 3 features most correlated with the target
5. Create scatter plots for top correlated features vs target
6. Check for outliers using box plots

**💡 Hint:** Use `.describe()`, `plt.hist()`, `sns.heatmap()`, and `sns.scatterplot()` on training data only.

In [ ]:
# Summary statistics (training set only)
print("Training Set Summary Statistics:")
print(X_train.describe())


In [ ]:
# Target variable distribution
plt.figure(figsize=(10, 5))
plt.hist(y_train, bins=50, edgecolor='black', alpha=0.7)
plt.xlabel('Median House Value')
plt.ylabel('Frequency')
plt.title('Target Variable Distribution (Training Set)')
plt.axvline(y_train.mean(), color='red', linestyle='--', label=f'Mean: {y_train.mean():.2f}')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Correlation analysis and heatmap
train_data = X_train.copy()
train_data['Target'] = y_train.values

corr_matrix = train_data.corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Heatmap (Training Set)')
plt.tight_layout()
plt.show()

# Top 3 features correlated with target
correlations = corr_matrix['Target'].drop('Target').sort_values(ascending=False)
print("\nTop 3 features correlated with target:")
for i, (feature, corr) in enumerate(correlations.head(3).items(), 1):
    print(f"  {i}. {feature}: {corr:.4f}")


In [ ]:
# Scatter plots for top features
top_features = correlations.head(3).index.tolist()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, feature in enumerate(top_features):
    axes[i].scatter(X_train[feature], y_train, alpha=0.3, s=10)
    axes[i].set_xlabel(feature)
    axes[i].set_ylabel('Target')
    axes[i].set_title(f'{feature} vs Target')

plt.tight_layout()
plt.show()


---
## 6. Baseline Model: Linear Regression

**Theory:**  
Linear Regression assumes a linear relationship between features and target. It's fast, interpretable, and serves as an excellent baseline. The model learns coefficients (weights) for each feature to minimize the sum of squared errors.

**What you need to do:**  
Train a Linear Regression model and evaluate it on the validation set.

**Tasks:**
1. Initialize the Linear Regression model
2. Train (fit) the model on training data
3. Make predictions on validation set
4. Calculate and display:
   - Mean Absolute Error (MAE)
   - Mean Squared Error (MSE)
   - Root Mean Squared Error (RMSE)
   - R² Score

**💡 Hint:** Use `.fit()`, `.predict()`, and metrics from `sklearn.metrics`.

In [ ]:
# Train Linear Regression model
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
print("Linear Regression model trained!")


In [ ]:
# Make predictions and calculate metrics
y_pred_lr = lr_model.predict(X_val)

lr_mae = mean_absolute_error(y_val, y_pred_lr)
lr_mse = mean_squared_error(y_val, y_pred_lr)
lr_rmse = np.sqrt(lr_mse)
lr_r2 = r2_score(y_val, y_pred_lr)

print("Linear Regression - Validation Performance:")
print(f"  MAE:  {lr_mae:.4f}")
print(f"  MSE:  {lr_mse:.4f}")
print(f"  RMSE: {lr_rmse:.4f}")
print(f"  R2:   {lr_r2:.4f}")


---
## 7. Cross-Validation for Linear Regression

**Theory:**  
Cross-validation provides a more robust estimate of model performance by training and evaluating the model multiple times on different subsets of data. K-Fold CV splits data into K folds, trains on K-1 folds, and validates on the remaining fold, rotating through all combinations.

**What you need to do:**  
Perform 5-fold cross-validation on the training set to get a better estimate of model performance.

**Tasks:**
1. Use `cross_val_score()` with 5 folds
2. Calculate RMSE for each fold (use `scoring='neg_mean_squared_error'` and take square root)
3. Display mean and standard deviation of CV scores

**💡 Hint:** `cross_val_score()` returns negative MSE, so you need to negate and take the square root. Use `scoring='neg_root_mean_squared_error'` if available.

In [ ]:
# Perform cross-validation
cv_scores = cross_val_score(lr_model, X_train, y_train, cv=5, scoring='neg_root_mean_squared_error')
cv_rmse = -cv_scores

print("Linear Regression - 5-Fold Cross-Validation:")
for i, score in enumerate(cv_rmse, 1):
    print(f"  Fold {i}: RMSE = {score:.4f}")
print(f"\nMean RMSE: {cv_rmse.mean():.4f} (+/- {cv_rmse.std():.4f})")


---
## 8. Tree-Based Model: Decision Tree Regressor

**Theory:**  
Decision Trees partition the feature space into regions through recursive binary splits. They can capture non-linear relationships and interactions between features without requiring feature scaling. However, they tend to overfit if not properly regularized.

**What you need to do:**  
Train a Decision Tree Regressor and compare its performance to Linear Regression.

**Tasks:**
1. Initialize a Decision Tree Regressor with `random_state=42`
2. Train on training data
3. Evaluate on validation set
4. Calculate the same metrics as Linear Regression
5. Compare performance to Linear Regression

**💡 Hint:** Without constraints, Decision Trees can perfectly memorize training data. We'll tune this in the next section.

In [ ]:
# Train Decision Tree model (unconstrained)
dt_model = DecisionTreeRegressor(random_state=42)
dt_model.fit(X_train, y_train)
print("Decision Tree model trained!")
print(f"Tree depth: {dt_model.get_depth()}")
print(f"Number of leaves: {dt_model.get_n_leaves()}")


In [ ]:
# Make predictions and calculate metrics
y_pred_dt = dt_model.predict(X_val)

dt_mae = mean_absolute_error(y_val, y_pred_dt)
dt_mse = mean_squared_error(y_val, y_pred_dt)
dt_rmse = np.sqrt(dt_mse)
dt_r2 = r2_score(y_val, y_pred_dt)

print("Decision Tree - Validation Performance:")
print(f"  MAE:  {dt_mae:.4f}")
print(f"  MSE:  {dt_mse:.4f}")
print(f"  RMSE: {dt_rmse:.4f}")
print(f"  R2:   {dt_r2:.4f}")

# Compare with Linear Regression
print("\nComparison with Linear Regression:")
print(f"  Linear Regression RMSE: {lr_rmse:.4f}")
print(f"  Decision Tree RMSE:     {dt_rmse:.4f}")


---
## 9. Cross-Validation for Decision Tree

**What you need to do:**  
Perform 5-fold cross-validation on the Decision Tree model.

**💡 Hint:** If CV scores vary significantly from validation scores, the model may be overfitting. This motivates hyperparameter tuning.

In [ ]:
# Perform cross-validation for Decision Tree
cv_scores_dt = cross_val_score(dt_model, X_train, y_train, cv=5, scoring='neg_root_mean_squared_error')
cv_rmse_dt = -cv_scores_dt

print("Decision Tree - 5-Fold Cross-Validation:")
for i, score in enumerate(cv_rmse_dt, 1):
    print(f"  Fold {i}: RMSE = {score:.4f}")
print(f"\nMean RMSE: {cv_rmse_dt.mean():.4f} (+/- {cv_rmse_dt.std():.4f})")

# Note about overfitting
if cv_rmse_dt.mean() > dt_rmse + 0.1:
    print("\nNote: CV performance is worse than validation, suggesting overfitting.")


---
## 10. Hyperparameter Tuning: Decision Tree

**Theory:**  
Hyperparameter tuning finds the optimal model configuration that balances bias and variance. For Decision Trees, key hyperparameters include:
- `max_depth`: Maximum tree depth (prevents overfitting)
- `min_samples_split`: Minimum samples required to split a node
- `min_samples_leaf`: Minimum samples required at leaf nodes
- `max_features`: Number of features to consider for each split

**What you need to do:**  
Use GridSearchCV to find the best hyperparameters for the Decision Tree.

**Tasks:**
1. Define a parameter grid with:
   - `max_depth`: [3, 5, 7, 10, None]
   - `min_samples_split`: [2, 5, 10]
   - `min_samples_leaf`: [1, 2, 4]
2. Use GridSearchCV with 5-fold CV
3. Fit on training data
4. Display best parameters and best CV score
5. Evaluate the best model on validation set

**💡 Hint:** Use `scoring='neg_mean_squared_error'` and set `n_jobs=-1` to use all CPU cores.

In [ ]:
# Define parameter grid and perform GridSearchCV
param_grid = {
    'max_depth': [3, 5, 7, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

grid_search = GridSearchCV(
    DecisionTreeRegressor(random_state=42),
    param_grid,
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)
print("GridSearchCV completed!")


In [ ]:
# Display best parameters and evaluate on validation set
print("Best parameters:", grid_search.best_params_)
print(f"Best CV RMSE: {-grid_search.best_score_:.4f}")

# Evaluate tuned model on validation set
best_dt = grid_search.best_estimator_
y_pred_dt_tuned = best_dt.predict(X_val)

dt_tuned_rmse = np.sqrt(mean_squared_error(y_val, y_pred_dt_tuned))
dt_tuned_mae = mean_absolute_error(y_val, y_pred_dt_tuned)
dt_tuned_r2 = r2_score(y_val, y_pred_dt_tuned)

print("\nTuned Decision Tree - Validation Performance:")
print(f"  MAE:  {dt_tuned_mae:.4f}")
print(f"  RMSE: {dt_tuned_rmse:.4f}")
print(f"  R2:   {dt_tuned_r2:.4f}")


---
## 11. Model Comparison

**What you need to do:**  
Create a summary comparison of all models tested.

**Tasks:**
1. Create a DataFrame or table comparing:
   - Linear Regression
   - Decision Tree (default)
   - Decision Tree (tuned)
2. Include metrics: RMSE, MAE, R²
3. Identify which model performs best on validation data

**💡 Hint:** Store all results in a dictionary and convert to a pandas DataFrame for clean visualization.

In [ ]:
# Create model comparison table
results = {
    'Model': ['Linear Regression', 'Decision Tree (Default)', 'Decision Tree (Tuned)'],
    'RMSE': [lr_rmse, dt_rmse, dt_tuned_rmse],
    'MAE': [lr_mae, dt_mae, dt_tuned_mae],
    'R2': [lr_r2, dt_r2, dt_tuned_r2]
}

results_df = pd.DataFrame(results)
results_df = results_df.sort_values('RMSE')

print("Model Comparison (Validation Set):")
print("="*60)
print(results_df.to_string(index=False))
print("="*60)

best_model_name = results_df.iloc[0]['Model']
print(f"\nBest model: {best_model_name}")


---
## 12. Final Evaluation on Test Set

**⚠️ CRITICAL: This is your ONE AND ONLY test set evaluation!**

**Theory:**  
The test set provides an unbiased estimate of how your model will perform on completely unseen data in production. This is your final report card. If you used the test set during development, this number would be artificially optimistic.

**What you need to do:**  
Evaluate your best model (from validation performance) on the held-out test set.

**Tasks:**
1. Select your best model based on validation performance
2. Make predictions on the test set
3. Calculate final metrics: RMSE, MAE, R²
4. Compare test set performance to validation performance
5. Create a scatter plot: Actual vs Predicted values
6. Display residuals distribution

**💡 Hint:** If test performance is significantly worse than validation, your model may have overfit to the validation set.

In [ ]:
# Final evaluation on test set
# Select best model (Decision Tree Tuned based on validation)
final_model = best_dt

# Predictions on test set
y_test_pred = final_model.predict(X_test)

# Calculate final metrics
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
test_mae = mean_absolute_error(y_test, y_test_pred)
test_r2 = r2_score(y_test, y_test_pred)

print("="*60)
print("FINAL TEST SET EVALUATION")
print("="*60)
print(f"Model: Decision Tree (Tuned)")
print(f"  RMSE: {test_rmse:.4f}")
print(f"  MAE:  {test_mae:.4f}")
print(f"  R2:   {test_r2:.4f}")
print("="*60)

# Compare with validation
print(f"\nValidation RMSE: {dt_tuned_rmse:.4f}")
print(f"Test RMSE:       {test_rmse:.4f}")


In [ ]:
# Visualize predictions vs actual values
plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_test_pred, alpha=0.5, s=20)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Actual Values')
plt.ylabel('Predicted Values')
plt.title('Predictions vs Actual Values (Test Set)')
plt.tight_layout()
plt.show()


In [ ]:
# Analyze residuals
residuals = y_test - y_test_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Residuals distribution
axes[0].hist(residuals, bins=50, edgecolor='black', alpha=0.7)
axes[0].axvline(0, color='red', linestyle='--')
axes[0].set_xlabel('Residuals')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Residuals Distribution')

# Residuals vs predicted
axes[1].scatter(y_test_pred, residuals, alpha=0.5, s=20)
axes[1].axhline(0, color='red', linestyle='--')
axes[1].set_xlabel('Predicted Values')
axes[1].set_ylabel('Residuals')
axes[1].set_title('Residuals vs Predicted')

plt.tight_layout()
plt.show()

print(f"Residuals - Mean: {residuals.mean():.4f}, Std: {residuals.std():.4f}")


---
## 13. Key Takeaways & Next Steps

**What you should have learned:**
1. ✅ Proper data splitting prevents data leakage
2. ✅ EDA helps understand data before modeling
3. ✅ Start with simple baselines (Linear Regression)
4. ✅ Cross-validation provides robust performance estimates
5. ✅ Hyperparameter tuning improves model performance
6. ✅ Test set evaluation gives final, unbiased performance

**Reflection Questions:**
- Which model performed better and why?
- How did hyperparameter tuning affect Decision Tree performance?
- What's the difference between validation and test set performance?
- Which features were most important for prediction?

---

### 🚀 Extension Activities

**This notebook structure is ready for plug-and-play with other models!**

Try replacing the Decision Tree with:
- **Random Forest Regressor** (ensemble of trees)
- **Gradient Boosting Regressor** (sequential boosting)
- **XGBoost Regressor** (optimized gradient boosting)
- **LightGBM Regressor** (fast gradient boosting)
- **Support Vector Regressor** (SVR)

For each new model:
1. Follow the same workflow (sections 8-10)
2. Use appropriate hyperparameters for that model
3. Compare results in section 11
4. Update final evaluation if it becomes the best model

---

**AI Tech Institute** | *Building Tomorrow's AI Engineers Today*